## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [2]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)


True

## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [3]:

# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-5.4-mini")

In [4]:
# running this pretty much returns a coroutine. Because under the hood asyncio is being used. 
# we need to use await 
Runner.run(agent, "Tell a joke about Autonomous AI Agents")

<coroutine object Runner.run at 0x105e3c040>

In [5]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")


In [6]:
print(result)

RunResult:
- Last agent: Agent(name="Jokester", ...)
- Final output (str):
    Autonomous AI agents are like interns who never sleep, never ask for lunch, and still somehow email the wrong person at 2 a.m.
- 1 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


In [7]:
# Here is the final output

print(result.final_output)

Autonomous AI agents are like interns who never sleep, never ask for lunch, and still somehow email the wrong person at 2 a.m.


In [8]:
# Here is the detail of the LLM calls

result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': 'msg_0fb2728a881578d1006a83450b547c8196919ad4c89569f047',
  'content': [{'annotations': [],
    'text': 'Autonomous AI agents are like interns who never sleep, never ask for lunch, and still somehow email the wrong person at 2 a.m.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

In [9]:
result._original_input
result.context_wrapper

RunContextWrapper(context=None, usage=Usage(requests=1, input_tokens=22, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=33, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=55, request_usage_entries=[RequestUsage(input_tokens=22, output_tokens=33, total_tokens=55, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens_details=OutputTokensDetails(reasoning_tokens=0))]), turn_input=[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'}], _approvals={}, tool_input=None)

## Adding Observability with a trace

In [10]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

Why did the Autonomous AI Agent bring a ladder to work?

Because it heard it needed to *escalate* issues on its own.


## Now go and look at the trace

https://platform.openai.com/traces

In [11]:
# Streaming
# according to OpenAI documentation they instruct us to use if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
# we can see why here; there are different event.types and if we try to use event.data.delta on the wrong type you will get an error
# example error; AttributeError: 'AgentUpdatedStreamEvent' object has no attribute 'data'

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    print(event.type)

agent_updated_stream_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_response_event
raw_

In [12]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

1. Why did the AI agent bring a ladder to work?  
   Because it wanted to reach the next level of thinking.

2. My AI agent said it could handle anything.  
   Then I asked it to schedule a meeting with “somebody.”  
   It’s still searching the entire internet.

3. Why was the AI agent bad at keeping secrets?  
   Because it had too many open contexts.

4. I told my AI agent to be more creative.  
   It replied, “Define creative.”

5. What do you call an AI agent that never takes breaks?  
   A model employee.

## Part 2: Adding a tool

In [13]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [14]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [15]:
push("HEY!!")

Push: HEY!!


In [16]:
push

<function __main__.push(message)>

In [17]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [18]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x11908f750>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

In [19]:
push_tool.params_json_schema

{'properties': {'message': {'title': 'Message', 'type': 'string'}},
 'required': ['message'],
 'title': 'push_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [20]:
push_tool.description

'Send the given message to the user as a push notification'

In [21]:

notifier = Agent(name="Notifier", model="gpt-5.4-mini", instructions="You notify the user upon request", tools=[push_tool])

In [22]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


Notified the user: “The pizza is here.”


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [ ]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

In [ ]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

## Memory approach 1 - just manually pass in the list of dicts

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

In [ ]:
response.to_input_list()

In [ ]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

In [ ]:
response = await Runner.run(agent, next_input)
print(response.final_output)

## Another approach - use OpenAI Agents SDK built in SQLLite session

In [23]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Ed.", session=session)
print(response.final_output)

In [ ]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>